**Question** How do the models perform on the held-out test split across all four prediction tasks?
**Method** Load `results/core_results.json` generated by code, recompute the evaluation table when needed, and show five fixed-seed repeats for the train-only classical model.
**Takeaway** The test split is reported once, with imbalance-aware metrics and an explicit majority baseline.

## Metric definitions
**F1** is the harmonic mean of precision and recall. **PR-AUC** is the area under the precision-recall curve. **MCC** is a correlation coefficient robust to class imbalance. **Top-k** counts a target if it appears among the k highest scores. **IoU** is intersection over union of predicted and true propagation node sets.

In [ ]:
from pathlib import Path
import sys, json, os
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd(); sys.path.extend([str(ROOT),str(ROOT/'notebooks'),str(ROOT/'python-ml')])
from notebooks.scripts.run_experiments import run
result=run(retrain=False, reduced=os.environ.get('AEGIS_REDUCED')=='1')
table=pd.DataFrame(result['models']).T
display(table[['model','f1','pr_auc','mcc','balanced_accuracy','root_top1','root_top3','propagation_iou','ttf_mae']])

### What this tells us
These are freshly computed held-out metrics, including the majority baseline. Accuracy alone is omitted from the headline because the nominal/failure mix makes a constant prediction deceptively competitive.

In [ ]:
repeats=pd.read_json(ROOT/'notebooks/results/repeat_results.json')
summary=repeats.groupby('model').agg({'f1':['mean','std'],'pr_auc':['mean','std'],'mcc':['mean','std']}).round(4)
display(summary)
repeats.groupby('model')['f1'].mean().plot(kind='bar',yerr=repeats.groupby('model')['f1'].std(),capsize=4,figsize=(6,4),title='Five-seed failure F1'); plt.ylabel('F1'); plt.show()

### What this tells us
The repeat table reports mean plus standard deviation over seeds 0–4 for the explicitly repeated train-only logistic baseline. A full five-seed deep-model comparison requires `RETRAIN=True` and is not silently substituted with copied historical values.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from notebooks.common.data import load_dataset, split_arrays
from notebooks.common.metrics import failure_metrics
# The runner's JSON stores the computed confusion matrix for each failure model.
for name,row in result['models'].items():
    if row['confusion_matrix']:
        print(name, row['confusion_matrix'])

### What this tells us
The confusion matrices make the cost of false negatives and false positives visible. Propagation IoU and root-cause top-k are separate tasks, so a strong failure detector is not automatically a strong explainer.